1 montagem do Drive, instalação de pacotes e verificação da GPU

In [ ]:
import os
import shutil
import zipfile
import torch
from google.colab import drive

drive.mount("/content/drive")

!pip install -q "rfdetr[train,loggers]" supervision mlflow

if not torch.cuda.is_available():
    raise RuntimeError("GPU não detectada: ative Runtime -> Change runtime type -> T4 GPU.")
print(f"GPU ativa: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
GPU ativa: NVIDIA L4


2 cópia e descompactação do dataset no disco local (/content)

In [ ]:
pasta_projeto_drive = "/content/drive/MyDrive/doutorado/20261 visao computacional/projeto/dataset"
caminho_zip_drive = os.path.join(pasta_projeto_drive, "kingdomino_pontuador.v1-v1.coco-segmentation.zip")

caminho_zip_local = "/content/dataset.zip"
pasta_dataset_local = "/content/dataset_tabuleiro"

if os.path.exists(pasta_dataset_local):
    shutil.rmtree(pasta_dataset_local)

print("copiando e descompactando dataset no disco local...")
shutil.copy(caminho_zip_drive, caminho_zip_local)
with zipfile.ZipFile(caminho_zip_local, "r") as arquivo_zip:
    arquivo_zip.extractall(pasta_dataset_local)

print("dataset pronto em:", pasta_dataset_local)

copiando e descompactando dataset no disco local...
dataset pronto em: /content/dataset_tabuleiro


3 treinamento do RF-DETR Seg com MLflow

In [ ]:
import gc
import os
import shutil
import torch
import mlflow
from rfdetr import RFDETRSegSmall

if "modelo" in globals():
    del modelo
gc.collect()
torch.cuda.empty_cache()

saida_treino_local = "/content/saida_treino"
artefatos_mlflow_local = "/content/mlruns"

for pasta in [saida_treino_local, artefatos_mlflow_local]:
    if os.path.exists(pasta):
        shutil.rmtree(pasta)

# Cria automaticamente a pasta exigida pelo MLflow
os.makedirs(artefatos_mlflow_local, exist_ok=True)

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_TRACKING_URI"] = f"file://{artefatos_mlflow_local}"
mlflow.set_tracking_uri(f"file://{artefatos_mlflow_local}")

modelo = RFDETRSegSmall(resolution=648)

modelo.train(
    dataset_dir=pasta_dataset_local,
    epochs=40,
    batch_size=8,
    grad_accum_steps=2,
    lr=1e-4,
    warmup_epochs=3.0,
    early_stopping=True,
    early_stopping_patience=8,
    early_stopping_use_ema=True,
    mlflow=True,
    project="kingdomino_ppgt13007",
    run="rfdetr_seg_small_648_bs8",
    output_dir=saida_treino_local
)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-18 02:05:14] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-seg-small.pt


/root/.roboflow/models/rf-detr-seg-small.pt:   0%|          | 0.00/129M [00:00<?, ?iB/s]

[2026-11-18 02:05:19] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-seg-small.pt
[2026-11-18 02:05:20] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-seg-small.pt already exists with correct MD5 hash.
[2026-11-18 02:05:22] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-seg-small.pt already exists with correct MD5 hash.


[2026-11-18 02:05:23] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 1. The detection head will be re-initialized to 1 classes.
INFO:pytorch_lightning.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.13/dist-packages/lightning_fabric/loggers/csv_logs.py:268: Experiment logs directory /content/saida_treino/ exists and is not empty. Previous log files in this directory will be deleted when the new ones are saved!


[2026-11-18 02:05:26] [INFO] rf-detr - Building Roboflow train dataset with square resize at resolution 648
[2026-11-18 02:05:26] [INFO] rf-detr - Using multi-scale training with square resize and scales: [768]
[2026-11-18 02:05:26] [INFO] rf-detr - Built 1 Albumentations transforms from config
[2026-11-18 02:05:26] [INFO] rf-detr - Built 1 Albumentations transforms from config
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[2026-11-18 02:05:26] [INFO] rf-detr - Skipping unannotated COCO grouping categories when assigning label indices: 0 (kingdomino-pontuador)
[2026-11-18 02:05:26] [INFO] rf-detr - Building Roboflow val dataset with square resize at resolution 648
[2026-11-18 02:05:26] [INFO] rf-detr - Using multi-scale training with square resize and scales: [768]
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


... (15 outputs left out)

INFO:pytorch_lightning.callbacks.early_stopping:Metric __rfdetr_effective_map__ improved by 0.007 >= min_delta = 0.001. New best score: 0.921


[2026-11-18 02:09:35] [INFO] rf-detr - Best EMA metric improved to 0.9213 (epoch 10)
[2026-11-18 02:09:57] [INFO] rf-detr - Best EMA metric improved to 0.9223 (epoch 11)


INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric __rfdetr_effective_map__ did not improve in the last 8 records. Best score: 0.921. Signaling Trainer to stop.


[2026-11-18 02:12:32] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.0000, ema=0.9223)


4 exportação dos pesos e métricas para o Google Drive

In [ ]:
import os
import shutil

pasta_projeto_drive = "/content/drive/MyDrive/doutorado/20261 visao computacional/projeto/dataset"
pasta_destino_drive = os.path.join(pasta_projeto_drive, "saida_treino_otimizada")
os.makedirs(pasta_destino_drive, exist_ok=True)

saida_treino_local = "/content/saida_treino"
artefatos_mlflow_local = "/content/mlruns"

# 1. Copia o melhor peso EMA para dentro de saida_treino_otimizada
melhor_peso_local = os.path.join(saida_treino_local, "checkpoint_best_ema.pth")
peso_destino_drive = os.path.join(pasta_destino_drive, "detector_tabuleiro_rfdetr.pth")

if os.path.exists(melhor_peso_local):
    shutil.copy2(melhor_peso_local, peso_destino_drive)
    tamanho_mb = os.path.getsize(peso_destino_drive) / (1024 * 1024)
    print(f"modelo copiado com sucesso: {peso_destino_drive} ({tamanho_mb:.2f} MB)")
else:
    print("aviso: checkpoint_best_ema.pth não encontrado em /content/saida_treino.")

# 2. Copia métricas e configurações
for arquivo in ["metrics.csv", "training_config.json", "hparams.yaml"]:
    origem = os.path.join(saida_treino_local, arquivo)
    if os.path.exists(origem):
        shutil.copy2(origem, os.path.join(pasta_destino_drive, arquivo))

# 3. Copia os artefatos do MLflow
destino_mlruns = os.path.join(pasta_destino_drive, "mlruns")
if os.path.exists(destino_mlruns):
    shutil.rmtree(destino_mlruns)
if os.path.exists(artefatos_mlflow_local):
    shutil.copytree(artefatos_mlflow_local, destino_mlruns)

print("arquivos atuais na pasta do drive:", os.listdir(pasta_destino_drive))

modelo copiado com sucesso: /content/drive/MyDrive/doutorado/20261 visao computacional/projeto/dataset/saida_treino_otimizada/detector_tabuleiro_rfdetr.pth (130.49 MB)
arquivos atuais na pasta do drive: ['metrics.csv', 'training_config.json', 'hparams.yaml', 'detector_tabuleiro_rfdetr.pth', 'mlruns']
